<a href="https://colab.research.google.com/github/Hussam780/Python-Projects/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## مشروع تصنيف الصور الشامل باستخدام TensorFlow

يغطي هذا الدليل العملي بناء نموذج تصنيف صور متكامل باستخدام مجموعة بيانات **CIFAR-10**. سنتبع منهجية هندسية تبدأ بتجهيز البيئة وتنتهي بتحليل النتائج.

### الخطوة 1: تهيئة البيئة واستيراد المكتبات
نبدأ باستيراد المكتبات الأساسية لمعالجة البيانات وبناء الشبكات العصبية الالتفافية (CNN).

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, callbacks
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

### الخطوة 2: تحميل ومعالجة البيانات (Data Preprocessing)
نقوم بتحميل البيانات، وتطبيع قيم البكسل (Normalization) لتكون بين 0 و 1، وتحويل التسميات إلى ترميز فئوي (One-Hot Encoding).

In [ ]:
(X_train, y_train), (X_test, y_test) = datasets.cifar10.load_data()

# تطبيع البيانات
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# الترميز الفئوي
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

### الخطوة 3: تصميم معمارية النموذج (Model Architecture)
سنقوم ببناء شبكة CNN عميقة تتضمن طبقات Dropout لمنع التجاوز (Overfitting) وطبقات BatchNormalization لاستقرار التدريب.

In [ ]:
model = models.Sequential([
    # الكتلة الأولى
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # الكتلة الثانية
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # الطبقات الكثيفة (Fully Connected)
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

### الخطوة 4: التدريب المتقدم (Training with Callbacks)
سنستخدم `EarlyStopping` لإيقاف التدريب تلقائياً إذا توقف النموذج عن التحسن، مما يوفر الوقت ويمنع الـ Overfitting.

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, y_train_cat,
                    epochs=50,
                    batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop])

### الخطوة 5: التقييم والتحليل (Evaluation & Analytics)
أهم جزء في أي مشروع هو فهم أين يخطئ النموذج. سنقوم برسم مصفوفة الارتباك (Confusion Matrix) لرؤية الفئات التي يختلط على النموذج التمييز بينها.

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = [np.argmax(element) for element in y_pred]

cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Truth')
plt.show()

## مشروع اكتشاف الوجوه (Face Detection) الشامل

تعتمد أنظمة رؤية الحاسوب الحديثة على شبكات عصبية مخصصة لاكتشاف الأنماط البشرية. سنقوم هنا ببناء نموذج مبسط لفهم كيفية تحديد موقع الوجه في الصورة.

### الخطوة 1: استيراد الأدوات المتقدمة
سنستخدم مكتبة `OpenCV` بجانب `TensorFlow` لتسهيل معالجة الصور الحقيقية.

In [ ]:
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

print("المكتبات جاهزة لبدء العمل على اكتشاف الوجوه.")

### الخطوة 2: فهم تقنيات اكتشاف الوجوه
هناك طريقتان رئيسيتان:
1. **Haar Cascades**: خوارزمية سريعة وتاريخية تعتمد على ميزات بسيطة.
2. **Deep Learning (CNN)**: تعتمد على شبكات عميقة مثل SSD أو MTCNN وهي الأكثر دقة.

سنستخدم هنا **Haar Cascade** المدمج مع OpenCV للبدء سريعاً، ثم نوضح كيف يتدرب نموذج التعلم العميق على ذلك.

In [ ]:
# تحميل مصنف الوجوه الجاهز من OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(img):
    # تحويل الصورة للون الرمادي لأن الخوارزمية لا تحتاج للألوان
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # اكتشاف الوجوه
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        # رسم مستطيل حول الوجه المكتشف
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

    return img

### الخطوة 3: التدريب العميق (Deep Learning Approach)
لبناء نموذج يتعرف على وجوه محددة (Face Recognition)، نحتاج إلى معمارية مثل **Siamese Networks** أو استخدام **Transfer Learning**.

إليك كيف نبني طبقة لاستخلاص ميزات الوجه:

In [ ]:
def build_face_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        # نستخدم Embedding Layer لتمثيل الوجه كمتجه رياضي
        layers.Dense(64, activation='l2_normalize')
    ])
    return model

face_encoder = build_face_model()
face_encoder.summary()

### الخطوة 4: التفسير الرياضي للنتائج
عندما يرى النموذج وجهاً، فإنه يحول الصورة إلى **متجه أرقام (Vector)**.
- إذا كانت المسافة بين متجهين صغيرة، فهما لنفس الشخص.
- إذا كانت المسافة كبيرة، فهما لشخصين مختلفين.

### الخطوة 5: التطبيق العملي
يمكنك الآن رفع صورة واختبار الكود عليها لاكتشاف الوجه وتحديده برمجياً.

## الدليل الشامل لمكتبة PyTorch

تتميز PyTorch بكونها تعتمد على الرسم البياني الحسابي الديناميكي (Dynamic Computational Graph)، مما يجعلها محببة جداً للمطورين والباحثين.

### الخطوة 1: تهيئة البيئة واستيراد PyTorch
نتأكد أولاً من توفر المكتبة واستخدام المعالج الرسومي (GPU) إذا كان متاحاً.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# التحقق من وجود GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Using Device: {device}")

### الخطوة 2: فهم الأساسيات (الموترات - Tensors)
الموتر هو الوحدة الأساسية في PyTorch، وهو يشبه مصفوفات NumPy لكن يمكنه العمل على الـ GPU.

In [ ]:
# إنشاء موتر بسيط
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# العمليات الحسابية وتتبع التدرجات (Autograd)
y = x ** 2 + 5

print("Tensor x:\n", x)
print("Result y:\n", y)

### الخطوة 3: بناء نموذج عصبي (Neural Network)
في PyTorch، نقوم بوراثة الكلاس `nn.Module` لتعريف طبقات النموذج.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # تعريف الطبقات
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32*32*3, 512) # دخل صورة ملونة
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 10) # 10 فئات للتصنيف

    def forward(self, x):
        # تحديد مسار البيانات (Forward Pass)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet().to(device)
print(model)

### الخطوة 4: حلقة التدريب (Training Loop)
خلافاً لـ TensorFlow، يتطلب PyTorch كتابة حلقة التدريب يدوياً، مما يعطيك تحكماً كاملاً.

In [ ]:
# تعريف دالة الخسارة والمحسن
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# مثال لهيكل حلقة التدريب
def train_one_epoch(data_loader):
    model.train()
    for batch, (X, y) in enumerate(data_loader):
        X, y = X.to(device), y.to(device)

        # 1. التصفير
        optimizer.zero_grad()

        # 2. التنبؤ
        pred = model(X)
        loss = criterion(pred, y)

        # 3. الاشتقاق العكسي (Backpropagation)
        loss.backward()

        # 4. تحديث الأوزان
        optimizer.step()

### الخلاصة
التعامل مع PyTorch يعتمد على:
1. **Tensors**: للبيانات.
2. **Modules**: للنماذج.
3. **Autograd**: لحساب التدرجات تلقائياً.
4. **Optimizers**: لتحديث الأوزان.